## VAMOS A IMPORTAR LAS DEPENDENCIAS PARA IR TRABAJANDO

In [18]:
from pathlib import Path
import pandas as pd

## Ahora vamos a caragar el dataset extraido

In [19]:
file_path = Path("data/staging/raw_sisben.parquet")

if not file_path.exists():
    # Probar subiendo un nivel por si el notebook está en una subcarpeta
    file_path = Path("../data/staging/raw_sisben.parquet")

print(f"Leyendo desde: {file_path.resolve()}")
df = pd.read_parquet(file_path)

print(
    f"\n Dataset cargado exitosamente: {df.shape[0]} filas x {df.shape[1]} columnas"
)

Leyendo desde: C:\Users\Titoelgato\OneDrive\Quinto semestre\ETL\Proyecto\Data\staging\raw_sisben.parquet

 Dataset cargado exitosamente: 5000 filas x 48 columnas


In [20]:
print("Dataset shape",df.shape)

display(df.head())

Dataset shape (5000, 48)


,cod_mpio,h_5,i1,i2,i3,i4,i5,i6,i7,i8,...,per011,per012,per013,per014,per015,per016,per017,per018,per019,per020
0,05001,1,1,0,1,1,0,0,1,1,...,9,1,9,9,1,2,2,2,5,99
1,05001,1,1,0,1,1,0,0,1,1,...,9,9,9,9,1,2,2,9,4,99
2,05001,0,1,0,0,0,0,0,0,1,...,9,9,9,9,1,2,2,2,2,10
3,05001,0,1,0,0,0,0,0,0,1,...,2,1,9,9,1,2,2,2,5,99
4,05001,0,1,0,0,0,0,0,0,1,...,9,9,9,9,1,2,2,2,2,10


In [22]:
print("municipios distintos:", df["cod_mpio"].nunique())
print(df["cod_mpio"].unique()[:10])
print(df.dtypes.value_counts())
print(df.isna().sum().loc[lambda s: s > 0])
print(df["corte"].value_counts(dropna=False))

municipios distintos: 819
<ArrowStringArray>
['05001', '18256', '50573', '20621', '17524', '15224', '52258', '88001',
 '73226', '27205']
Length: 10, dtype: str
str    48
Name: count, dtype: int64
Series([], dtype: int64)
corte
SIV_2022    5000
Name: count, dtype: int64


In [23]:
cols_texto = ["cod_mpio", "grupo", "clasificacion", "llave", "corte"]
num = df.drop(columns=cols_texto).apply(pd.to_numeric, errors="coerce")

print("valores no convertibles a número:", int(num.isna().sum().sum()))
print("\nfex:")
print(num["fex"].describe())
print("\ngrupo:\n", df["grupo"].value_counts(dropna=False))
print("\nzona:\n", df["zona"].value_counts(dropna=False))
print("\nlongitud de cod_mpio:", df["cod_mpio"].str.len().value_counts().to_dict())

llave_persona = ["cod_mpio", "llave", "hogar", "orden"]
print("\nllaves de persona duplicadas en la muestra:", int(df.duplicated(subset=llave_persona).sum()))

cols_per = [c for c in num.columns if c.startswith("per")]
pct_99 = (num[cols_per] == 99).mean().mul(100).round(1)
print("\n% de valores 99 (centinela) por columna per:")
print(pct_99[pct_99 > 0])

valores no convertibles a número: 0

fex:
count    5000.000000
mean      241.579807
std       344.572535
min         1.078740
25%         3.770060
50%        23.954887
75%       761.196393
max      1842.667820
Name: fex, dtype: float64

grupo:
 grupo
C    1871
B    1416
A     966
D     747
Name: count, dtype: int64

zona:
 zona
2    2608
1    2392
Name: count, dtype: int64

longitud de cod_mpio: {5: 5000}

llaves de persona duplicadas en la muestra: 0

% de valores 99 (centinela) por columna per:
per005b    68.5
per006b    58.0
per020     71.2
dtype: float64


In [24]:
df["fex_n"] = pd.to_numeric(df["fex"])
print(df.groupby("grupo")["fex_n"].agg(["count", "mean", "median", "min", "max"]).round(1))

       count   mean  median  min     max
grupo                                   
A        966   74.3     3.4  1.1   761.2
B       1416  192.8    24.0  1.1   761.2
C       1871  321.9    24.0  1.2  1842.7
D        747  349.3    24.0  1.1   761.2
